# Tutorial 11-1: The Apriori Engine - Pruning and Candidate Generation
**Course:** CSEN 140: Data Mining/Machine Learning  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
This tutorial demonstrates the internal mechanics of the **Apriori Algorithm**. We will focus on how the algorithm uses the **Anti-Monotone Property** of support to reduce the number of itemsets it needs to count, moving from a brute-force approach toward an efficient candidate-based approach.

### Key Concepts
* **Apriori Principle:** If an itemset is frequent, all of its subsets must also be frequent. If a subset is infrequent, the superset is automatically infrequent.
* **Anti-monotone Property:** For all sets X and Y, if X is a subset of Y, the support of X will be greater than or equal to that of Y.
* **Candidate Generation ($F_{k-1} \times F_{k-1}$):** A method to generate potential $k$-itemsets by merging frequent $(k-1)$-itemsets that share a common prefix.
* **Support Counting:** Scanning the database to determine the actual frequency of candidate itemsets.

In [ ]:
# The transaction dataset from the lecture slides (Slide 5)
dataset = [
    {'Bread', 'Milk'},
    {'Bread', 'Diaper', 'Beer', 'Eggs'},
    {'Milk', 'Diaper', 'Beer', 'Coke'},
    {'Bread', 'Milk', 'Diaper', 'Beer'},
    {'Bread', 'Milk', 'Diaper', 'Coke'}
]

minsup_count = 3  # Minimum support threshold
print(f"Total Transactions: {len(dataset)}")
print(f"Minimum Support Count Threshold: {minsup_count}")

### 1. Frequent 1-Itemsets ($F_1$)
We begin by counting individual items. Only those meeting the `minsup_count` are kept.

In [ ]:
from collections import defaultdict

def get_frequent_1_itemsets(transactions, min_support):
    counts = defaultdict(int)
    for transaction in transactions:
        for item in transaction:
            counts[frozenset([item])] += 1
    
    return {itemset: count for itemset, count in counts.items() if count >= min_support}

F1 = get_frequent_1_itemsets(dataset, minsup_count)
print("Frequent 1-itemsets (F1):")
for itemset, count in sorted(F1.items()):
    print(f"{set(itemset)}: {count}")

### 2. Candidate Generation ($F_{k-1} \times F_{k-1}$)
We generate candidate $k$-itemsets by joining frequent $(k-1)$-itemsets. Following the lecture logic, we merge them if their first $(k-2)$ items are identical.

In [ ]:
def generate_candidates(prev_frequent_itemsets, k):
    """Generates C_k candidates from F_{k-1}"""
    candidates = set()
    # Sort items within itemsets to facilitate prefix matching
    list_itemsets = sorted([sorted(list(i)) for i in prev_frequent_itemsets])
    
    for i in range(len(list_itemsets)):
        for j in range(i + 1, len(list_itemsets)):
            # Join if first k-2 items are the same (Slide 31)
            l1 = list_itemsets[i][:k-2]
            l2 = list_itemsets[j][:k-2]
            if l1 == l2:
                candidates.add(frozenset(list_itemsets[i]) | frozenset(list_itemsets[j]))
    return candidates

C2 = generate_candidates(F1.keys(), 2)
print(f"Generated {len(C2)} raw candidates for C2:")
for c in sorted(C2): print(set(c))

### 3. Candidate Pruning
We immediately prune candidates if any of their $(k-1)$ subsets are not frequent. This is the core efficiency of Apriori.

In [ ]:
import itertools

def prune_candidates(candidates, prev_frequent_itemsets, k):
    """Removes candidates with infrequent subsets"""
    pruned_list = set()
    for candidate in candidates:
        is_valid = True
        # Check every subset of length k-1
        subsets = itertools.combinations(candidate, k-1)
        for s in subsets:
            if frozenset(s) not in prev_frequent_itemsets:
                is_valid = False
                break
        if is_valid:
            pruned_list.add(candidate)
    return pruned_list

C2_pruned = prune_candidates(C2, F1.keys(), 2)
print(f"Candidates remaining after pruning: {len(C2_pruned)}")

### 4. Support Counting and Elimination
We perform a single scan of the database to determine which candidates meet the minimum support.

In [ ]:
def count_support(transactions, candidates, min_support):
    counts = defaultdict(int)
    for transaction in transactions:
        for candidate in candidates:
            if candidate.issubset(transaction):
                counts[candidate] += 1
    
    return {itemset: count for itemset, count in counts.items() if count >= min_support}

F2 = count_support(dataset, C2_pruned, minsup_count)
print("Final Frequent 2-itemsets (F2):")
for itemset, count in sorted(F2.items()):
    print(f"{set(itemset)}: {count}")

## Summary & Conclusion

In this tutorial, we implemented the **Apriori Algorithm's** level-by-level exploration:
1. **Efficiency Gain:** By using the Anti-Monotone property, we avoid the brute-force complexity of checking all $2^d$ possible itemsets.
2. **Prefix-based Joining:** The $F_{k-1} \times F_{k-1}$ method systematically explores the pattern lattice.
3. **Pruning Power:** We saw how pruning candidates *before* support counting saves significant computational time, especially as the number of items $d$ increases.

**Next Steps:** In Tutorial 11-2, we will look at generating **Association Rules** (Support and Confidence) from these frequent itemsets.